# CosyVoice 3 Dialogue — Phase B (audio)

**Phase A (local):** `python run_script_agents.py your_script.txt` → `lines.json`

**Upload to Drive `inputs/`:**
1. `lines.json`
2. `male_ref.wav` + `female_ref.wav`
3. `ref_texts.json` (optional for CosyVoice instruct2; kept for compatibility)

Backend: Fun-CosyVoice3-0.5B — identity ref + emotion instruct (not emotion wav banks).

In [ ]:
import torch
assert torch.cuda.is_available(), "Runtime > Change runtime type > T4 GPU"
print(torch.cuda.get_device_name(0))
!nvidia-smi

In [ ]:
from pathlib import Path
from google.colab import drive

drive.mount("/content/drive")

RUN_NAME = "chokhi_dhani_cosy_v1"
ROOT = Path("/content/drive/MyDrive/voice-generation") / RUN_NAME
INPUT_DIR = ROOT / "inputs"
CLIPS_DIR = ROOT / "clips"
OUT_DIR = ROOT / "outputs"
for d in (INPUT_DIR, CLIPS_DIR, OUT_DIR):
    d.mkdir(parents=True, exist_ok=True)
print("ROOT =", ROOT)
print("inputs:", sorted(p.name for p in INPUT_DIR.iterdir()))

In [ ]:
# Install CosyVoice (lean Colab deps — NOT full requirements.txt)
# Full reqs pin old torch + tensorrt/deepspeed and hang/fight Colab.
from pathlib import Path
from huggingface_hub import snapshot_download

COSY_REPO = Path("/content/CosyVoice")
MODEL_DIR = Path("/content/Fun-CosyVoice3-0.5B")

if not (COSY_REPO / ".git").exists():
    !git clone --recursive https://github.com/FunAudioLLM/CosyVoice.git {COSY_REPO}
else:
    print("CosyVoice repo ready")

# Keep Colab's torch; only install inference libs
# Pin protobuf (5.x has runtime_version; 4.x breaks modern transformers).
# If proto/transformers errors persist: Runtime > Restart session, then re-run this cell.
!pip install -q --force-reinstall "protobuf==5.29.3"
!pip install -q \
  "transformers==4.51.3" openai-whisper lightning HyperPyYAML omegaconf hydra-core inflect \
  librosa soundfile wetext wget gdown modelscope "diffusers==0.29.0" x-transformers conformer \
  pyworld onnx onnxruntime "protobuf==5.29.3" pydantic rich pydub huggingface_hub
# If you still see proto/transformers import errors: Runtime > Restart session, then re-run from this cell.

# Download weights if missing
has_weights = any(MODEL_DIR.glob("*.pt")) or (MODEL_DIR / "cosyvoice3.yaml").exists()
if not has_weights:
    print("Downloading Fun-CosyVoice3-0.5B-2512...")
    snapshot_download("FunAudioLLM/Fun-CosyVoice3-0.5B-2512", local_dir=str(MODEL_DIR))
else:
    print("Model already present")

print("repo:", COSY_REPO)
print("model:", MODEL_DIR)
print("weights:", sorted(p.name for p in MODEL_DIR.glob("*"))[:12])


In [ ]:
import json
from google.colab import files

needed = ["lines.json", "male_ref.wav", "female_ref.wav"]
have = {p.name for p in INPUT_DIR.iterdir()}
missing = [n for n in needed if n not in have]
if missing:
    print("Upload:", missing)
    uploaded = files.upload()
    for name, data in uploaded.items():
        (INPUT_DIR / name).write_bytes(data)

with open(INPUT_DIR / "lines.json", encoding="utf-8") as f:
    lines = json.load(f)
print(f"{len(lines)} lines")
for L in lines[:3]:
    print(f"  [{L['index']}] {L['speaker']} ({L.get('emotion')}) {L['text'][:70]}...")

In [ ]:
import sys
import torch

sys.path.insert(0, str(COSY_REPO))
sys.path.insert(0, str(COSY_REPO / "third_party" / "Matcha-TTS"))

from cosyvoice.cli.cosyvoice import AutoModel

# fp16=False + force float32: avoids Float vs BFloat16 crash with newer stacks
cosy = AutoModel(model_dir=str(MODEL_DIR), fp16=False)
cosy.model.llm.float()
cosy.model.flow.float()
cosy.model.hift.float()
print("sample_rate", cosy.sample_rate)
print("llm dtype", next(cosy.model.llm.parameters()).dtype)

In [ ]:
EMOTION_INSTRUCT = {
    "neutral": "warm natural conversational tone, speak a bit faster",
    "curious": "curious questioning tone, speak a bit faster",
    "playful": "playful amused tone, lively pace",
    "excited": "excited but not shouting, lively pace",
    "warm": "warm friendly tone, conversational pace",
    "wonder": "soft wonder, still conversational — not slow",
    "explanatory": "clear friendly tone, conversational pace",
}

def cosy_instruct(emotion, speaker="M", emphasize=None):
    # Keep SHORT — long English instruct gets spoken aloud by CosyVoice.
    # Identity comes from ref wav, not from instruct prose.
    emo = (emotion or "warm").lower()
    style = EMOTION_INSTRUCT.get(emo, EMOTION_INSTRUCT["warm"])
    parts = [f"Speak in English with a {style}."]
    words = [w.strip() for w in (emphasize or []) if w and str(w).strip()]
    if words:
        quoted = ", ".join(f'"{w}"' for w in words[:2])
        parts.append(f"Emphasize {quoted}.")
    return "You are a helpful assistant. " + " ".join(parts) + "<|endofprompt|>"

def ref_for(speaker):
    name = "male_ref.wav" if speaker == "M" else "female_ref.wav"
    path = INPUT_DIR / name
    assert path.exists(), path
    return path


In [ ]:
import sys
import re
import subprocess
import torch
import torchaudio
from pydub import AudioSegment
from pydub.silence import detect_nonsilent

# Load model if you skipped the AutoModel cell (or runtime restarted)
if "cosy" not in globals():
    sys.path.insert(0, str(COSY_REPO))
    sys.path.insert(0, str(COSY_REPO / "third_party" / "Matcha-TTS"))
    from cosyvoice.cli.cosyvoice import AutoModel
    cosy = AutoModel(model_dir=str(MODEL_DIR), fp16=False)
    cosy.model.llm.float()
    cosy.model.flow.float()
    cosy.model.hift.float()
    print("loaded cosy, sample_rate", cosy.sample_rate)

# Conversational podcast targets (higher than careful narration)
TARGET_WPM = {
    "wonder": 155, "warm": 170, "explanatory": 170, "curious": 165,
    "neutral": 170, "playful": 185, "excited": 195,
}
# CosyVoice native speed (1.0 = default; >1 = faster)
SPEED_BY_EMOTION = {
    "wonder": 1.08, "warm": 1.10, "explanatory": 1.10, "curious": 1.10,
    "neutral": 1.10, "playful": 1.12, "excited": 1.15,
}
TOLERANCE, MIN_F, MAX_F = 0.10, 0.85, 1.15

def _words(text):
    return len(re.findall(r"[A-Za-z0-9][A-Za-z0-9'’-]*", text))

def _nonsilent_sec(path):
    audio = AudioSegment.from_wav(path)
    ranges = detect_nonsilent(audio, min_silence_len=250, silence_thresh=-40)
    if not ranges:
        return len(audio) / 1000.0
    return sum((e - s) for s, e in ranges) / 1000.0

def adjust_pace(path, text, emotion):
    words = _words(text)
    if words < 4:
        return
    dur = _nonsilent_sec(path)
    if dur < 0.4:
        return
    actual = words / (dur / 60.0)
    target = TARGET_WPM.get(emotion, 170)
    ratio = actual / target
    if abs(ratio - 1.0) <= TOLERANCE:
        print(f"  pace ok ~{actual:.0f} wpm (target {target})")
        return
    factor = max(MIN_F, min(MAX_F, target / actual))
    tmp = path.with_suffix(".atempo_tmp.wav")
    subprocess.run(
        ["ffmpeg", "-y", "-i", str(path), "-filter:a", f"atempo={factor:.4f}", str(tmp)],
        check=True, capture_output=True,
    )
    tmp.replace(path)
    print(f"  pace x{factor:.3f} ~{actual:.0f}->{target} wpm")

FORCE_REGEN = False  # True to overwrite existing clips

for L in lines:
    out = CLIPS_DIR / f"{L['index']:03d}_{L['speaker']}.wav"
    if out.exists() and not FORCE_REGEN:
        print(f"skip {out.name}")
        continue
    emotion = (L.get("emotion") or "warm").lower()
    ref = ref_for(L["speaker"])
    emph = L.get("emphasize") or []
    instruct = cosy_instruct(emotion, speaker=L["speaker"], emphasize=emph)
    speed = SPEED_BY_EMOTION.get(emotion, 1.10)
    print(f"[{L['index']}] {L['speaker']}/{emotion} speed={speed} emph={emph} <- {ref.name}")
    chunks = list(
        cosy.inference_instruct2(
            L["text"], instruct, str(ref), stream=False, speed=speed,
        )
    )
    audio = torch.cat([c["tts_speech"] for c in chunks], dim=1)
    torchaudio.save(str(out), audio.cpu(), cosy.sample_rate)
    adjust_pace(out, L["text"], emotion)
    print("  ->", out.name)

print("clips:", len(list(CLIPS_DIR.glob("*.wav"))))


In [ ]:
# Stitch + loudnorm
from pydub import AudioSegment
from pydub.silence import detect_nonsilent
import subprocess

def trim_silence(audio, silence_thresh=-40, min_silence_len=250, padding_ms=50):
    nonsilent = detect_nonsilent(audio, min_silence_len=min_silence_len, silence_thresh=silence_thresh)
    if not nonsilent:
        return audio
    start = max(nonsilent[0][0] - padding_ms, 0)
    end = min(nonsilent[-1][1] + padding_ms, len(audio))
    return audio[start:end]

episode = AudioSegment.empty()
for L in lines:
    path = CLIPS_DIR / f"{L['index']:03d}_{L['speaker']}.wav"
    clip = trim_silence(AudioSegment.from_wav(path))
    pb = int(L.get("pause_before_ms") or 0)
    pa = int(L.get("pause_after_ms") or 350)
    if pb:
        episode += AudioSegment.silent(duration=pb)
    episode += clip
    episode += AudioSegment.silent(duration=pa)

raw_wav = OUT_DIR / "episode_raw.wav"
final_mp3 = OUT_DIR / "final.mp3"
episode.export(raw_wav, format="wav")
subprocess.run([
    "ffmpeg", "-y", "-i", str(raw_wav),
    "-af", "loudnorm=I=-16:TP=-1.5:LRA=11",
    str(final_mp3),
], check=True)
print("Wrote", final_mp3)